In [10]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time
from cuml.neighbors import KNeighborsClassifier
from cuml.metrics import mean_squared_error, mean_squared_log_error, median_absolute_error, r2_score, accuracy_score, confusion_matrix, kl_divergence
from cuml.metrics import log_loss, roc_auc_score, nan_euclidean_distances, pairwise_distances, sparse_pairwise_distances
from cuml.model_selection import train_test_split, KFold

In [11]:
df = cudf.read_csv('heart_disease_health_indicators_BRFSS2015.csv')
df

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [12]:
from ipynb.fs.full.Normalization_to_import import *

In [13]:
from ipynb.fs.full.Data_Generation_to_import import *

In [28]:
class Neighours(object):
    def __init__(self, dataset, generated_data_global_1):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.X = cudf.DataFrame(self.dataset.copy().drop(['HeartDiseaseorAttack'], axis = 1))
        self.y = cudf.DataFrame(self.dataset['HeartDiseaseorAttack'].copy())
        self.generated_data_global_1 = generated_data_global_1.copy().reset_index(drop = True)
        self.dataset_numpy_array = self.dataset.copy().to_numpy()
        self.X_cupy_array = self.X.to_cupy()
        self.y_cupy_array = self.y.to_cupy()
        self.gen_data_cupy_array = self.generated_data_global_1.to_cupy()

    def train_test_split(self):
        global X_train_global
        global X_test_global
        global y_train_global
        global y_test_global

        X_train, X_test, y_train, y_test = train_test_split(self.X_cupy_array, self.y_cupy_array, test_size=0.2, random_state=42)
        X_train_global = X_train
        X_test_global = X_test
        y_train_global = y_train
        y_test_global = y_test

    def KNeighborsClassifier(self):
        global knc_global
        global knc_predict_test_global
        global knc_predict_global_1
        global mean_squared_error_global_knc
        global mean_squared_log_error_global_knc
        global median_absolute_error_global_knc
        global r2_score_global_knc
        global accuracy_score_global_knc
        global kl_divergence_global_knc
        global log_loss_global_knc
        global roc_auc_score_global_knc
        global nan_euclidean_distances_global_knc
        global pairwise_distances_global_knc
        global sparse_pairwise_distances_global_knc

        model = KNeighborsClassifier(weights='uniform', verbose=False, output_type=None)
        knc = model.fit(X_train_global.copy(), y_train_global.copy().ravel())
        knc_global = knc
        
        preds_test = model.predict(X_test_global.copy())
        knc_predict_test_global = cudf.DataFrame(preds_test.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True)
        
        mean_squared_error_global_knc = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_knc = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_knc = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_knc = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_knc = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_knc = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_knc = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_knc = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_knc = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_knc = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_knc = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model.predict(self.gen_data_cupy_array)
        knc_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['HeartDiseaseorAttack']).reset_index(drop = True) 

    def main(self):
        st = time.time()
        self.train_test_split()
        self.KNeighborsClassifier()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [29]:
class Metrics(object):
    def KNeighborsClassifier(self):
        print('KNeighborsClassifier: ')
        print('Mean squared error: ')
        print(mean_squared_error_global_knc)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_knc)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_knc)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_knc)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_knc)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_knc)
        print('\n')

    def main(self):
        self.KNeighborsClassifier()

In [30]:
neighbours = Neighours(df, generated_data_global)
neighbours.main()
metrics = Metrics()
print("Metrics for not normalized data is ready. Run 'metrics.main()'!")
#metrics.main()

Execution time: 4.816184997558594 seconds
Metrics for not normalized data is ready. Run 'metrics.main()'!


In [31]:
knc_predict_global_1

,HeartDiseaseorAttack
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
999995,0.0
999996,0.0
999997,0.0
999998,0.0


In [32]:
knc_predict_global_1.max()

HeartDiseaseorAttack    1.0
dtype: float64

In [33]:
metrics.main()

KNeighborsClassifier: 
Mean squared error: 
0.10300378429517502


Median Absolute Error: 
0.0


R2 Score: 
-0.20974482978529574


Accuracy Score: 
0.896996215704825


Kl Divergence: 
inf


ROC AUC Score: 
0.5428584814071655


